# 📈 Notebook 04 — Model Evaluation
**CCS3440 Artificial Intelligence | SmartCare AI Risk Prediction**

---
### 🎯 Objectives (Task 06)
- Evaluate all trained models on the test set
- Generate: **Confusion Matrix, ROC Curve, Classification Report**
- Compare models and select the best performer
- Save evaluation results to `reports/`

---

## 1️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')
print('✅ Libraries imported')

## 2️⃣ Load Data & Models

In [ ]:
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

MODEL_NAMES = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost']

models = {}
for name in MODEL_NAMES:
    models[name] = joblib.load(f'../models/{name}.pkl')
    print(f'✅ Loaded: {name}')

## 3️⃣ Evaluate All Models

In [ ]:
results = []

for name, model in models.items():
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1 Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results)
print('=== Evaluation Results ===')
results_df

## 4️⃣ ROC Curves — All Models

In [ ]:
plt.figure(figsize=(9, 7))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves.png', dpi=150)
plt.show()
print('✅ ROC curve saved')

## 5️⃣ Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'{name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices.png', dpi=150)
plt.show()
print('✅ Confusion matrices saved')

## 6️⃣ Save Evaluation Results

In [ ]:
results_df.to_csv('../reports/evaluation_results.csv', index=False)
print('✅ Saved: reports/evaluation_results.csv')
results_df.sort_values('ROC-AUC', ascending=False)